In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [3]:
# 1. Project paths and settings

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = PROJECT_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "NVDA"]


In [4]:
# 2. Helper functions

def find_statement_value(statement, possible_names, period):
    """Find a financial metric in a statement for one period."""

    if statement is None or statement.empty:
        return None

    for name in possible_names:
        if name in statement.index:
            value = statement.loc[name, period]

            if pd.notna(value):
                return float(value)

    return None


def latest_two_values(statement, possible_names):
    """Return the latest and previous annual values."""

    if statement is None or statement.empty:
        return None, None, None

    periods = sorted(statement.columns, key=lambda x: pd.Timestamp(x))

    latest_period = periods[-1]
    previous_period = periods[-2] if len(periods) >= 2 else None

    latest = find_statement_value(
        statement, possible_names, latest_period
    )

    previous = (
        find_statement_value(statement, possible_names, previous_period)
        if previous_period is not None
        else None
    )

    return latest_period, latest, previous


def calculate_growth(latest, previous):
    """Calculate year-over-year growth when valid."""

    if latest is None or previous is None or previous == 0:
        return None

    return (latest - previous) / abs(previous) * 100


def safe_info_value(info, key):
    """Return a usable company information value."""

    value = info.get(key)

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return value


# Common financial statement labels used by Yahoo Finance.
REVENUE_NAMES = ["Total Revenue", "Operating Revenue", "Revenue"]
NET_INCOME_NAMES = ["Net Income", "Net Income Common Stockholders"]
DEBT_NAMES = ["Total Debt"]
CASH_NAMES = [
    "Cash And Cash Equivalents",
    "Cash Cash Equivalents And Short Term Investments",
    "Cash And Cash Equivalents And Short Term Investments",
]


In [5]:
# 3. Download financial statements and company metrics

summary_rows = []
income_statement_rows = []
balance_sheet_rows = []
chart_rows = []

for ticker in TICKERS:
    print(f"Downloading financial data for {ticker}...")

    try:
        company = yf.Ticker(ticker)

        income = company.financials
        balance = company.balance_sheet

        try:
            info = company.info or {}
        except Exception as error:
            print(f"Company information unavailable for {ticker}: {error}")
            info = {}

        # Save annual income statement rows in a tidy format.
        if income is not None and not income.empty:
            for period in income.columns:
                for metric in income.index:
                    value = income.loc[metric, period]

                    income_statement_rows.append({
                        "Ticker": ticker,
                        "Period_End": pd.Timestamp(period).date(),
                        "Metric": metric,
                        "Value": value,
                    })

        # Save annual balance sheet rows in a tidy format.
        if balance is not None and not balance.empty:
            for period in balance.columns:
                for metric in balance.index:
                    value = balance.loc[metric, period]

                    balance_sheet_rows.append({
                        "Ticker": ticker,
                        "Period_End": pd.Timestamp(period).date(),
                        "Metric": metric,
                        "Value": value,
                    })

        # Get latest and previous annual revenue.
        revenue_period, revenue, previous_revenue = latest_two_values(
            income, REVENUE_NAMES
        )

        # Get latest and previous annual net income.
        _, net_income, previous_net_income = latest_two_values(
            income, NET_INCOME_NAMES
        )

        profit_margin = (
            net_income / revenue * 100
            if revenue and net_income is not None
            else None
        )

        revenue_growth = calculate_growth(revenue, previous_revenue)
        net_income_growth = calculate_growth(
            net_income, previous_net_income
        )

        # Get the latest balance sheet values.
        balance_period = None
        total_debt = None
        cash = None

        if balance is not None and not balance.empty:
            balance_period = max(
                balance.columns, key=lambda x: pd.Timestamp(x)
            )

            total_debt = find_statement_value(
                balance, DEBT_NAMES, balance_period
            )

            cash = find_statement_value(
                balance, CASH_NAMES, balance_period
            )

        # Get market valuation information.
        summary_rows.append({
            "Ticker": ticker,
            "Latest_Income_Period": (
                pd.Timestamp(revenue_period).date()
                if revenue_period is not None else None
            ),
            "Revenue": revenue,
            "Previous_Year_Revenue": previous_revenue,
            "Revenue_Growth_Pct": revenue_growth,
            "Net_Income": net_income,
            "Previous_Year_Net_Income": previous_net_income,
            "Net_Income_Growth_Pct": net_income_growth,
            "Net_Profit_Margin_Pct": profit_margin,
            "Market_Cap": safe_info_value(info, "marketCap"),
            "Trailing_PE": safe_info_value(info, "trailingPE"),
            "Forward_PE": safe_info_value(info, "forwardPE"),
            "Total_Debt": total_debt,
            "Cash_And_Short_Term_Investments": cash,
        })

        # Prepare annual chart data.
        if income is not None and not income.empty:
            periods = sorted(
                income.columns, key=lambda x: pd.Timestamp(x)
            )

            for period in periods:
                period_revenue = find_statement_value(
                    income, REVENUE_NAMES, period
                )
                period_net_income = find_statement_value(
                    income, NET_INCOME_NAMES, period
                )

                chart_rows.append({
                    "Ticker": ticker,
                    "Fiscal_Period_End": pd.Timestamp(period),
                    "Revenue": period_revenue,
                    "Net_Income": period_net_income,
                })

    except Exception as error:
        print(f"Could not process {ticker}: {error}")



In [6]:
# 4. Save the financial data

summary = pd.DataFrame(summary_rows)
income_data = pd.DataFrame(income_statement_rows)
balance_data = pd.DataFrame(balance_sheet_rows)
chart_data = pd.DataFrame(chart_rows)

summary_file = REPORTS_DIR / "fundamental_summary.csv"
income_file = REPORTS_DIR / "annual_income_statements.csv"
balance_file = REPORTS_DIR / "annual_balance_sheets.csv"
chart_file = REPORTS_DIR / "annual_fundamental_trends.csv"

summary.to_csv(summary_file, index=False)
income_data.to_csv(income_file, index=False)
balance_data.to_csv(balance_file, index=False)
chart_data.to_csv(chart_file, index=False)


In [7]:
# 5. Create a revenue comparison chart

if not chart_data.empty:
    revenue_data = chart_data.dropna(subset=["Revenue"])

    if not revenue_data.empty:
        fig, ax = plt.subplots(figsize=(11, 6))

        for ticker, group in revenue_data.groupby("Ticker"):
            group = group.sort_values("Fiscal_Period_End")

            ax.plot(
                group["Fiscal_Period_End"],
                group["Revenue"] / 1_000_000_000,
                marker="o",
                linewidth=2,
                label=ticker,
            )

        ax.set_title(
            "Annual Revenue Comparison",
            fontsize=16,
            fontweight="bold",
        )
        ax.set_xlabel("Fiscal period end")
        ax.set_ylabel("Revenue (USD billions)")
        ax.grid(True, alpha=0.25)
        ax.legend(title="Stock")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        fig.tight_layout()
        fig.savefig(
            FIGURES_DIR / "annual_revenue_comparison.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig)

    # Create a net income comparison chart.
    income_data_for_chart = chart_data.dropna(subset=["Net_Income"])

    if not income_data_for_chart.empty:
        fig, ax = plt.subplots(figsize=(11, 6))

        for ticker, group in income_data_for_chart.groupby("Ticker"):
            group = group.sort_values("Fiscal_Period_End")

            ax.plot(
                group["Fiscal_Period_End"],
                group["Net_Income"] / 1_000_000_000,
                marker="o",
                linewidth=2,
                label=ticker,
            )

        ax.set_title(
            "Annual Net Income Comparison",
            fontsize=16,
            fontweight="bold",
        )
        ax.set_xlabel("Fiscal period end")
        ax.set_ylabel("Net income (USD billions)")
        ax.grid(True, alpha=0.25)
        ax.legend(title="Stock")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        fig.tight_layout()
        fig.savefig(
            FIGURES_DIR / "annual_net_income_comparison.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig)



In [8]:
# 6. Display the summary

print("\nFundamental analysis completed.")
print(f"Summary saved to: {summary_file}")
print(f"Income statements saved to: {income_file}")
print(f"Balance sheets saved to: {balance_file}")
print(f"Annual trends saved to: {chart_file}")

if not summary.empty:
    print("\nFundamental summary:")
    print(summary.round(2).to_string(index=False))
else:
    print("No summary data was returned. Check your internet connection.")



Fundamental analysis completed.
Summary saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/fundamental_summary.csv
Income statements saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/annual_income_statements.csv
Balance sheets saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/annual_balance_sheets.csv
Annual trends saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/annual_fundamental_trends.csv

Fundamental summary:
Ticker Latest_Income_Period      Revenue  Previous_Year_Revenue  Revenue_Growth_Pct   Net_Income  Previous_Year_Net_Income  Net_Income_Growth_Pct  Net_Profit_Margin_Pct    Market_Cap  Trailing_PE  Forward_PE   Total_Debt  Cash_And_Short_Term_Investments
  AAPL           2025-09-30 4.161610e+11           3.910350e+11                6.43 1.120100e+11              9.373600e+10                  19.50                  26.92 5081985449984        39.44       36.38